In [1]:
# Import Libraries

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [10]:
# Transform: convert to tensor + normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download MNIST dataset
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Data loaders
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = torch.utils.data.DataLoader(test_data, batch_size=1000, shuffle=False)

In [11]:
# Define the Model
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3), nn.ReLU(),
            nn.MaxPool2d(2), nn.Flatten(),
            nn.Linear(32*13*13, 10)
        )
    def forward(self, x):
        return self.net(x)

model = DigitNet()

In [12]:
# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
#Training Loop
for epoch in range(5):  # Train for 5 epochs
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.0655
Epoch 2, Loss: 0.0249
Epoch 3, Loss: 0.0050
Epoch 4, Loss: 0.1218
Epoch 5, Loss: 0.0309


In [14]:
# Evaluation
model.eval()
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()

accuracy = 100. * correct / len(test_loader.dataset)
print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 98.01%


In [8]:
# Get one batch of images and labels
data_iter = iter(test_loader)
images, labels = next(data_iter)

# Run the model on this batch
model.eval()
with torch.no_grad():
    outputs = model(images)
    preds = outputs.argmax(dim=1)

# Compare predictions vs actual labels
print("Predicted:", preds[:10].tolist())
print("Actual:   ", labels[:10].tolist())


Predicted: [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
Actual:    [7, 2, 1, 0, 4, 1, 4, 9, 5, 9]
